[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb)

# Lab 2: Trace Your Agent

### Instrument the agent with OpenTelemetry and inspect its traces in Phoenix

In notebook 1, you built the agent. Here, OpenInference captures LLM calls automatically,
manual spans make agent and tool steps visible, and every trace goes directly to the shared
Phoenix board on Fly.

The workshop's canonical agent remains in `utils.py`; this notebook attaches a tracer to
that implementation rather than creating a second agent that could drift.


---
## 1. Install the lightweight Phoenix tracing SDKs

Use the OTEL and client SDKs rather than installing a second Phoenix server inside Colab.
The shared server is already running; these packages only export spans and query them.


In [ ]:
%pip install -q arize-phoenix-otel==0.17.1 arize-phoenix-client==3.7.0 openinference-instrumentation-openai==0.1.76

> Install packages before importing the agent or tracing SDK. If Colab says the
> runtime needs a restart, restart it now and rerun the notebook from the top.


---
## Bootstrap the workshop agent

Refresh stale `utils.py` copies, load the synthetic ledger, and connect through the same
workshop gateway and seat key as notebook 1. The shared Phoenix board requires a separate
participant `PHOENIX_API_KEY` supplied by the organizer; it is not your LiteLLM workshop
key. Save it as a Colab secret named `PHOENIX_API_KEY` or enter it at the hidden prompt.


In [ ]:
import urllib.request
import os
import importlib
from pathlib import Path

utils_path = Path("utils.py")
if not utils_path.exists() or "def set_span_tracer(tracer, attributes=None):" not in utils_path.read_text():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils
utils = importlib.reload(utils)

utils.load_data()
utils.connect()
SEAT = utils.seat_tag()
PHOENIX_BOARD_URL = "https://fmlds-eval-workshop-phoenix.fly.dev"
PHOENIX_PROJECT = "00-Part-2-tracing-set-up"
PHOENIX_API_KEY = utils.get_secret("PHOENIX_API_KEY")
if not PHOENIX_API_KEY:
    from getpass import getpass
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
from phoenix.client import Client
phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
try:
    phoenix_client.projects.list()
except Exception as exc:
    if "401" not in str(exc) and "Unauthorized" not in str(exc):
        raise
    from getpass import getpass
    print("Saved Phoenix key was rejected. Enter the participant board key supplied by the organizer.")
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
    phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
    phoenix_client.projects.list()
from utils import NAIVE_SYSTEM_PROMPT, run_agent

print("bootstrap done | seat tag:", SEAT, "| shared Phoenix project:", PHOENIX_PROJECT)


## 2. Send traces to the shared Phoenix board

Register the shared OTLP collector and authenticate with the board key. The whole room
uses the `00-Part-2-tracing-set-up` project, which sorts before the gateway project.


In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name=PHOENIX_PROJECT,
    endpoint=f"{PHOENIX_BOARD_URL}/v1/traces",
    protocol="http/protobuf",
    batch=False,
    api_key=PHOENIX_API_KEY,
)
print("Shared Phoenix project:", PHOENIX_PROJECT)
print("Phoenix board:", PHOENIX_BOARD_URL)


## 3. Instrument LLM calls and attach the agent tracer

`OpenAIInstrumentor` creates an LLM span for each OpenAI SDK call. The canonical agent
adds the surrounding spans: an **AGENT** root, a **CHAIN** for each router call, a nested
LLM span, then a tool-handling **CHAIN** containing one **TOOL** span per tool execution.
Inputs, outputs, and tool errors are recorded as span attributes.


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.semconv.trace import SpanAttributes

openai_instrumentor = OpenAIInstrumentor()
if openai_instrumentor.is_instrumented_by_opentelemetry:
    openai_instrumentor.uninstrument()
openai_instrumentor.instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(
    tracer_provider.get_tracer(__name__),
    attributes={SpanAttributes.SESSION_ID: SEAT},
)
print("OpenAI calls instrumented; manual agent/chain/tool spans enabled")


> **Trace data note:** spans contain prompts and tool inputs/outputs. This workshop uses
> synthetic transactions; do not send personal or otherwise sensitive data to a tracer.


---
## 4. Challenge 1: signed expenses versus cash flow

Ask the agent for October expenses without explaining that income and refunds are positive
while expenses are negative. In the trace, inspect the generated pandas code and tool output.


In [ ]:
gross_october = run_agent(
    "Use analyze_transactions to calculate my total expenses in October and report the dollar amount.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/october-gross-expenses",
)
print(gross_october["answer"])


Compare the answer with two plausible calculations: the positive magnitude of
negative transactions, and the signed total of every October transaction.


In [ ]:
october = df[df["date"].dt.month == 10]
truth_gross_expenses = -october.loc[october["amount"] < 0, "amount"].sum()
signed_cash_flow = october["amount"].sum()
print(f"expense magnitude: {truth_gross_expenses:,.2f}")
print(f"signed cash flow:  {signed_cash_flow:,.2f}")
print(f"positive income/refunds included: {october.loc[october['amount'] > 0, 'amount'].sum():,.2f}")


### Inspect this run's tool I/O

The Phoenix trace shows the overall span tree. This helper also prints the exact tool
arguments and results returned to the agent for this individual run.


In [ ]:
def show_tool_io(result):
    "Print tool arguments and returned results from one agent run."
    for message in result["messages"]:
        if message.get("role") == "assistant":
            for call in message.get("tool_calls") or []:
                function = call["function"]
                print(f"Calling {function['name']}: {function['arguments']}")
        elif message.get("role") == "tool":
            print("Tool result:", message["content"])


In [ ]:
show_tool_io(gross_october)

### Harder sign case: net expenses after refunds

Gross expenses count negative transactions. Net expenses subtract refunds but must exclude
income. The synthetic ledger contains an October refund.


In [ ]:
net_october = run_agent(
    "Use analyze_transactions to calculate net expenses in October after transaction refunds, excluding income.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/net-expenses-after-refunds",
)
print(net_october["answer"])


In [ ]:
refunds = october.loc[
    october["raw_description"].str.contains("REFUND", case=False) & (october["amount"] > 0), "amount"
].sum()
print(f"gross expenses: {truth_gross_expenses:,.2f}")
print(f"refunds:        {refunds:,.2f}")
print(f"net expenses:   {truth_gross_expenses - refunds:,.2f}")


In [ ]:
show_tool_io(net_october)

---
## 5. Challenge 2: merchant aliases and a decoy

Descriptions use both `STARBUCKS` and the alias `SBUX`. Tully's is another coffee shop,
not Starbucks. Inspect the tool input and see whether the generated filter handles both cases.


In [ ]:
starbucks_run = run_agent(
    "Use analyze_transactions to find my total Starbucks spending and report the dollar amount, even if it is zero.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/starbucks-aliases",
)
print(starbucks_run["answer"])


In [ ]:
is_starbucks = df["raw_description"].str.contains(r"STARBUCKS|SBUX", case=False, regex=True)
truth_starbucks = -df.loc[is_starbucks & (df["amount"] < 0), "amount"].sum()
tully_decoy_count = df["raw_description"].str.contains("TULLY'S", case=False).sum()
print(f"Starbucks + SBUX expense magnitude: {truth_starbucks:,.2f}")
print(f"Tully's decoy rows to exclude:      {tully_decoy_count}")


In [ ]:
show_tool_io(starbucks_run)

---
## 6. Challenge 3: what happens when the data has no requested field?

The ledger has no merchant-location column. Ask for a location breakdown, then inspect the
schema and tool response rather than assuming what the model will say.


In [ ]:
missing_location = run_agent(
    "Show me my spending broken down by merchant location.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/missing-location",
)
print(missing_location["answer"])


In [ ]:
print("merchant_city exists:", "merchant_city" in df.columns)
print("available columns:", sorted(df.columns.tolist()))


In [ ]:
show_tool_io(missing_location)

### Find your runs in the shared Part 2 project

All participants share one Phoenix project, so use the `session.id` attribute to identify
your spans. Expand an AGENT span to inspect its nested router, LLM, tool, and tool-generated
LLM spans; compare that tree with the printed tool I/O above.


In [ ]:
spans_df = phoenix_client.spans.get_spans_dataframe(project_identifier=PHOENIX_PROJECT, limit=1000)
trace_columns = [
    column for column in (
        "name", "span_kind", "parent_id", "context.trace_id",
        "attributes.session.id", "attributes.input.value", "attributes.output.value",
        "attributes.llm.token_count.prompt", "attributes.llm.token_count.completion",
    )
    if column in spans_df.columns
]
trace_id_column = next((column for column in ("context.trace_id", "trace_id") if column in spans_df.columns), None)
if trace_id_column and "attributes.session.id" in spans_df.columns:
    my_trace_ids = spans_df.loc[spans_df["attributes.session.id"] == SEAT, trace_id_column].dropna().unique()
    my_spans = spans_df[spans_df[trace_id_column].isin(my_trace_ids)]
else:
    my_spans = spans_df.tail(40)
display(my_spans[trace_columns])


---
## All traces go to the shared board

There is no separate publish step: notebook 2's agent, OpenAI, chain, and tool traces
arrive in the shared `00-Part-2-tracing-set-up` project. Agent spans carry your `session.id`
so the room can distinguish participant runs from one another and from gateway traces.


---
### Continue the workshop

[Next: Notebook 3 — Evaluations](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)
